In [0]:
# ============================================================
# Bronze — Source 11: ERP Export
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=11_erp/
# Target: bronze.erp catalog in Unity Catalog
#
# Tables:
#   - bronze.erp.orders
#
# Raw format: JSON / XML (daily partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "11_erp"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=18/")

In [0]:
# Cell 2 — Read JSON files and inspect schema
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path_json = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/*.json"

df_json = spark.read.format("json") \
    .load(path_json) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"JSON rows: {df_json.count()}")
df_json.printSchema()

In [0]:
# Cell 3 — Explode invoices array and flatten
from pyspark.sql.functions import explode

invoices_flat = df_json \
    .select(explode(col("invoices")).alias("invoice")) \
    .select(
        col("invoice.invoice_number"),
        col("invoice.order_id"),
        col("invoice.invoice_date"),
        col("invoice.due_date"),
        col("invoice.status"),
        col("invoice.currency"),
        col("invoice.subtotal_pence"),
        col("invoice.tax_pence"),
        col("invoice.cost_centre"),
        col("invoice.gl_account"),
        col("invoice.payment_terms"),
        col("invoice._source").alias("source"),
        col("invoice._format").alias("format"),
    )

print(f"Flattened invoices: {invoices_flat.count()} rows")
invoices_flat.show(3)

In [0]:
# Cell 4 — Read XML files
# Spark reads XML via spark-xml library (built into Databricks runtime)
path_xml = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/*.xml"

df_xml = spark.read.format("xml") \
    .option("rowTag", "invoice") \
    .load(path_xml) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"XML rows: {df_xml.count()}")
df_xml.printSchema()

In [0]:
# Cell 5 — Read all XML files and union with JSON
xml_files = []
for day in range(17, 25):
    try:
        files = dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day={day}/")
        for f in files:
            if f.name.endswith(".xml"):
                xml_files.append(f.path)
    except:
        pass

print(f"Found {len(xml_files)} XML files")

# Read all XML files
from functools import reduce
from pyspark.sql import DataFrame

dfs_xml = []
for path in xml_files:
    df = spark.read.format("xml") \
        .option("rowTag", "Invoice") \
        .load(path) \
        .filter(col("_metadata.file_modification_time") > lit(watermark))
    dfs_xml.append(df)

df_xml_all = reduce(DataFrame.union, dfs_xml)

# Normalise XML to match JSON flat schema
xml_flat = df_xml_all.select(
    col("invoice_number"),
    col("order_id"),
    col("invoice_date").cast("string"),
    col("due_date").cast("string"),
    col("status"),
    col("currency"),
    col("subtotal_pence"),
    col("tax_pence"),
    col("total_pence"),
    col("cost_centre"),
    col("gl_account"),
    col("payment_terms"),
    col("vendor_id"),
    col("vendor_name"),
    col("_source").alias("source"),
    col("_format").alias("format"),
)

print(f"XML rows: {xml_flat.count()}")

In [0]:
# Cell 6 — Add missing fields to JSON flat and union with XML
from pyspark.sql.functions import lit

# Add missing fields to JSON flat (total_pence, vendor_id, vendor_name)
invoices_json_full = invoices_flat.select(
    col("invoice_number"),
    col("order_id"),
    col("invoice_date"),
    col("due_date"),
    col("status"),
    col("currency"),
    col("subtotal_pence"),
    col("tax_pence"),
    lit(None).cast("long").alias("total_pence"),
    col("cost_centre"),
    col("gl_account"),
    col("payment_terms"),
    lit(None).cast("string").alias("vendor_id"),
    lit(None).cast("string").alias("vendor_name"),
    col("source"),
    col("format"),
)

# Union JSON + XML
df_combined = invoices_json_full.union(xml_flat)
print(f"Combined rows: {df_combined.count()}")
df_combined.show(3)

In [0]:
# Cell 7 — Write to Bronze and update watermark
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.erp")

df_combined.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.erp.invoices")

latest_ts = df_json.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, df_combined.count())
print(f"✅ bronze.erp.invoices: {df_combined.count()} rows (JSON + XML)")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.erp.invoices").collect()[0]['cnt']
print(f"bronze.erp.invoices: {count} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '11_erp'").show()